# 06 — Évaluation end-to-end et API

**Objectif** : vérifier le chemin complet, les métriques métier, la latence et le contrat HTTP.

**Critère de passage** : aucune fausse complétude, assertions citées et endpoints conformes.

In [ ]:
from pathlib import Path
import statistics
import sys
import time

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table

ROOT = bootstrap()
sys.path.insert(0, str(ROOT / 'backend' / 'evals'))
from run_evals import evaluate
from app.domain.models import QuestionRequest, ScopeSelection
from app.services.engine import engine

In [ ]:
report = await evaluate()
metrics = report['metrics']
assert metrics['false_completeness_rate'] == 0.0
assert metrics['citation_precision'] == 1.0
display_table([{'metric': name, 'value': value} for name, value in metrics.items()])

In [ ]:
display_table([
    {
        'case': result['id'],
        'profil_ok': result['profile_ok'],
        'statut_attendu': result['expected_status'],
        'statut_obtenu': result['actual_status'],
        'champs_couverts': ', '.join(result['covered_fields']),
    }
    for result in report['results']
])

In [ ]:
request = QuestionRequest(
    question='Quels indicateurs clients et opérationnels sont publiés pour 2025 ?',
    mode='deep',
    scope=ScopeSelection(document_ids=['foyer_annual_report_2025']),
)
measurements = []
for _ in range(10):
    started = time.perf_counter()
    result = await engine.answer(request)
    measurements.append((time.perf_counter() - started) * 1000)
ordered = sorted(measurements)
{
    'iterations': len(measurements),
    'p50_ms': round(statistics.median(measurements), 2),
    'p95_ms': round(ordered[min(len(ordered) - 1, int(len(ordered) * 0.95))], 2),
    'status': result.status,
}

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
health = client.get('/api/health')
answer = client.post('/api/query', json={
    'question': 'Quels chiffres clés sont publiés pour Global Health ?',
    'mode': 'deep',
    'scope': {'document_ids': ['foyer_annual_report_2025']},
})
assert health.status_code == 200
assert answer.status_code == 200
assert answer.json()['status'] == 'COMPLETE'
{
    'health': health.json(),
    'query_status': answer.json()['status'],
    'claims': len(answer.json()['claims']),
}

### Interprétation responsable

Un jeu initial de six questions valide le câblage du démonstrateur ; il ne prouve pas une qualité de production. Étendre ensuite les évaluations par type de document, entité, période, tableau, ambiguïté et question non répondable.